In [1]:
import sys
sys.path.append("../")

from src.llm.models import Model
from src.llm.loading import Loader
from pathlib import Path
from src.llm.prompt import system_prompt

/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = Model(name="Qwen/Qwen2.5-1.5B-Instruct")
tokenizer = model.tokenizer

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 5426.98it/s]


In [3]:
model.model = model.peft_model("../model")
model.name = f"{model.name} (fine-tuned)"

In [4]:
loader = Loader(path=Path("../data/data.json"), test_k=0.2, seed=42)
train_dataset, tests = loader.train_test_split()
train_dataset, tests

(Dataset({
     features: ['input', 'target'],
     num_rows: 39
 }),
 Dataset({
     features: ['input', 'target'],
     num_rows: 10
 }))

In [5]:
def build_prompt(input: str) -> str:
    prompt_messages = [
            {"role": "system", "content": system_prompt.render()},
            {"role": "user", "content": input},
        ]
    return tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

In [6]:
# generate output

result = []
for test in tests:
    output = model.generate(build_prompt(test["input"]))
    result.append({
        "input": test["input"],
        "prediction": output,
        "target": test["target"],
    })

len(result)

10

In [7]:
# evaluate
from src.evaluating import Evaluator

evaluator = Evaluator(
    predictions=[row["prediction"] for row in result],
    references=[row["target"] for row in result],
)

metrics = evaluator.evaluate()
metrics

{'samples': 10,
 'exact_match': 0.7,
 'masking_recall': 0.8235294117647058,
 'over_masking_rate': 0.15151515151515152,
 'text_preservation': 0.9990704611275861}

In [8]:
# generate report
from src.reporting import write_baseline_report

report_path = write_baseline_report(
    model_name=model.name,
    metrics=metrics,
    predictions=result,
    output_path="../reports/05_llm_fine_tuned.md",
    preview_size=len(result)
)

print(f"Fine-Tuned report saved to {report_path}")

preview = result[:3]
for index, row in enumerate(preview, start=1):
    print()
    print(f"Sample {index}")
    print("INPUT:", row["input"])
    print("PREDICTION:", row["prediction"])
    print("TARGET:", row["target"])

Fine-Tuned report saved to ../reports/05_llm_fine_tuned.md

Sample 1
INPUT: Hello Mrs. Walker, can you check why my recent transfer of $250 hasn't been processed? My account number is 567890 and my email is megan.walker@example.com.
PREDICTION: Hello [PREFIX], can you check why my recent transfer of [AMOUNT] hasn't been processed? My account number is [ACCOUNT_NUMBER] and my email is [EMAIL].
TARGET: Hello [PREFIX], can you check why my recent transfer of [AMOUNT] hasn't been processed? My account number is [ACCOUNT_NUMBER] and my email is [EMAIL].

Sample 2
INPUT: Dear Ms. Garcia, I need to report a lost debit card. My phone number is (555) 789-0123 and my account number is 678901.
PREDICTION: Dear [PREFIX], I need to report a lost debit card. My phone number is [PHONE_NUMBER] and my account number is [ACCOUNT_NUMBER].
TARGET: Dear [PREFIX], I need to report a lost debit card. My phone number is [PHONE_NUMBER] and my account number is [ACCOUNT_NUMBER].

Sample 3
INPUT: I'd like to kno